# Lecture 4: Becoming a Backprop Ninja — 初始化、BN 与训练诊断

跟随 Karpathy 的 [makemore](https://github.com/karpathy/makemore) 系列第 4 讲，深入探索**为什么初始化很重要**以及如何用 Batch Normalization 和训练诊断工具让深层网络稳定训练。

**学习路线：**
1. 浅层 MLP + 手动实现 Batch Normalization
2. 初始化诊断：Tanh 饱和、softmax 过度自信、loss 曲线
3. BN 的推理模式：EMA running statistics
4. Kaiming 初始化原理 & 用类封装的深层网络
5. 训练诊断四件套：激活分布、梯度分布、权重梯度、update/data 比率

In [ ]:
import torch
import torch.nn.functional as F          # 常用函数：cross_entropy, softmax 等
import matplotlib.pyplot as plt           # 绘图库，用于可视化 loss 曲线和直方图
%matplotlib inline

In [ ]:
# ============================================================
# 读取数据集：每行一个英文名字
# ============================================================
words = open('D:\\Vault-4\\Projects\\makemore\\names.txt', 'r').read().splitlines()
words[:8]                                # 预览前 8 个名字

In [ ]:
len(words)                               # 数据集中名字的总数（约 32K）

In [ ]:
# ============================================================
# 构建字符词表 & 字符↔整数的映射表
# 26 个字母 + 1 个特殊符号 '.'（起始/终止标记）→ vocab_size = 27
# ============================================================
chars = sorted(list(set(''.join(words)))) # 取所有出现过的字符并排序
stoi = {s:i+1 for i,s in enumerate(chars)}# 字符→索引（a=1, b=2, ..., z=26）
stoi['.'] = 0                             # '.' 特殊标记 → 索引 0
itos = {i:s for s,i in stoi.items()}      # 索引→字符（反向映射）
vocab_size = len(itos)                    # 词表大小 = 27
print(f'{vocab_size} unique characters: {itos}')

In [ ]:
# ============================================================
# 构建数据集：将名字转为 (context → next_char) 的训练样本
# block_size=3 表示用前 3 个字符预测下一个字符
# ============================================================
block_size = 3  # 上下文长度：用几个字符来预测下一个

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size       # 初始上下文全填 '.'（索引 0）
        for ch in w + '.':               # 遍历名字的每个字符 + 终止符
            ix = stoi[ch]                # 当前字符的索引
            X.append(context)            # 输入：当前上下文窗口
            Y.append(ix)                 # 标签：下一个字符的索引
            context = context[1:] + [ix] # 滑动窗口：去掉最早的，加入当前字符
    X = torch.tensor(X)                  # shape: (N, block_size)
    Y = torch.tensor(Y)                  # shape: (N,)
    print(X.shape, Y.shape)
    return X, Y

# ============================================================
# 按 80/10/10 比例划分训练集/验证集/测试集
# ============================================================
import random
random.seed(42)
random.shuffle(words)                    # 打乱名字顺序
n1 = int(0.8*len(words))                # 训练集截止位置
n2 = int(0.9*len(words))                # 验证集截止位置
Xtr, Ytr = build_dataset(words[:n1])     # 训练集（80%）
Xdev, Ydev = build_dataset(words[n1:n2]) # 验证集（10%）
Xte, Yte = build_dataset(words[n2:])     # 测试集（10%）

print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)
print(Xte.shape, Yte.shape)

## 1. 浅层 MLP + 手动 Batch Normalization

**概念解释：** 这一部分在第 3 讲的 MLP 基础上，加入 Batch Normalization（BN）来稳定训练。上一讲我们发现两个问题：(1) Tanh 神经元在训练初期大量饱和，梯度消失；(2) 初始 loss 远大于理论值 3.296（softmax 过度自信）。BN 正是解决这些问题的关键技术。

**直觉理解：** BN 的核心思想非常简单——在每一层的线性变换之后、激活函数之前，把数据"拉回"到均值 0、标准差 1 的范围。为什么？因为 Tanh 的"甜区"（梯度较大的区域）恰好在 0 附近。如果输入值偏离太远（比如全是 5 或 -5），Tanh 输出就会卡在 ±1，梯度几乎为零，网络就"学不动"了。

$$\hat{x} = \gamma \cdot \frac{x - \mu_B}{\sigma_B} + \beta$$

逐项解读：
- $\mu_B, \sigma_B$ = 当前 mini-batch 的均值和标准差（沿 batch 维度计算）
- 减均值、除标准差 → 标准化到 N(0,1)
- $\gamma$（bngain）和 $\beta$（bnbias）→ 可学习的缩放和偏移，让网络自己决定最终分布
- 为什么标准化后还要缩放/偏移？因为强制 N(0,1) 可能限制表达能力——$\gamma, \beta$ 让网络在需要时"逃出"标准化的约束

**生活类比：** BN 就像老师在批改作文前先把所有学生的分数标准化——把平均分调到 60 分，标准差调到 10 分。这样不管是哪个班交上来的（不同 mini-batch），评判标准都是一致的。然后 $\gamma$ 和 $\beta$ 就像最终的"难度系数"和"起评分"，由系统自动学习调整。

In [ ]:
# ============================================================
# 定义浅层 MLP + 手动 Batch Normalization 参数
# 架构：Embedding(27→10) → Linear(30→200) → BN → Tanh → Linear(200→27)
# ============================================================
n_embd = 10                              # 字符嵌入维度
n_hidden = 200                           # 隐藏层神经元数量

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)  # 嵌入矩阵 (27, 10)

# W1 权重用 Kaiming 初始化：除以 sqrt(fan_in)，再乘 5/3 补偿 tanh 的方差压缩
W1 = torch.randn((block_size*n_embd, n_hidden), generator=g) * (5/3) / ((block_size*n_embd)**0.5)
# b1 被移除了——因为 BN 会先减去均值，偏置项会被直接抵消，加了也白加
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01  # 输出层权重，初始化很小 → 初始 logits 接近 0 → 均匀分布
b2 = torch.randn(vocab_size, generator=g) * 0                 # 输出层偏置，初始化为 0

# BN 的可学习参数：缩放因子 γ 和偏移量 β
bngain = torch.ones((1, n_hidden))       # γ，初始 = 1（不缩放）
bnbias = torch.zeros((1, n_hidden))      # β，初始 = 0（不偏移）

# 指数移动平均（EMA）维护的全局统计量，供推理时替代 mini-batch 统计量
# 初始值：mean=0, std=1（标准正态假设）
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2]             # 注意：b1 已移除，bngain/bnbias 也在列表中
for p in parameters:
    p.requires_grad = True
print(sum(p.nelement() for p in parameters), 'parameters')

## Batch Normalization（批归一化）

训练循环中这一行是手动实现的 BN：
```python
hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
```

### 公式拆解

对 `hpreact` 形状 $(B, n\_hidden)$，沿 batch 维度（`dim=0`）计算：

$$\mu_j = \frac{1}{B}\sum_{i=1}^{B} x_{ij}, \quad \sigma_j = \sqrt{\frac{1}{B}\sum_{i=1}^{B}(x_{ij}-\mu_j)^2}$$

$$\hat{x}_{ij} = \gamma_j \cdot \frac{x_{ij} - \mu_j}{\sigma_j} + \beta_j$$

其中 $\gamma$ = `bngain`（初始 1），$\beta$ = `bnbias`（初始 0）。

### 目的

1. **防止 tanh 饱和**：将 `hpreact` 拉回均值 0、标准差 1 的范围，让大部分值落在 tanh 的线性区（约 $[-1,1]$），避免梯度消失
2. **解耦层间依赖**：前面层参数变化不会让后面层输入分布漂移（Internal Covariate Shift）
3. **降低对初始化的敏感度**：即使初始化不佳，BN 也会自动把分布拉正
4. **$\gamma, \beta$ 保留表达能力**：标准化后再通过可学习参数缩放/偏移，网络可以自己决定最终分布

### Mini-batch 噪声 → 免费的正则化

因为 $\mu, \sigma$ 是在**随机抽取的 mini-batch**（这里 batch_size=32）上计算的，不是全局精确统计量，所以同一个样本在不同 batch 中会得到略不同的标准化结果。这种"抖动"等价于给网络注入了噪声，迫使网络学习对扰动鲁棒的特征，天然起到 **正则化** 效果（类似 Dropout）。batch 越小噪声越大。

> 推理时不能用 mini-batch 统计量（否则预测会抖动），需要用训练期间的全局 running mean/std。

In [ ]:
# ============================================================
# 训练循环：手动实现 Batch Normalization 的前向传播
# ============================================================
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    # ---- 1. 构造 mini-batch ----
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]           # 随机抽取 batch_size 个样本

    # ---- 2. 前向传播 ----
    emb = C[Xb]                          # 字符嵌入查表 → (32, 3, 10)
    embcat = emb.view(emb.shape[0], -1)  # 拼接 3 个嵌入向量 → (32, 30)
    hpreact = embcat @ W1                # 线性变换 → (32, 200)，不加 b1（BN 会减均值）

    # ---- 3. 手动 Batch Normalization ----
    # 沿 batch 维度（dim=0）计算当前 mini-batch 的均值和标准差
    bnmeani = hpreact.mean(0, keepdim=True)   # shape: (1, 200)
    bnstdi = hpreact.std(0, keepdim=True)     # shape: (1, 200)
    # 标准化 + 可学习的缩放和偏移
    hpreact = bngain * (hpreact - bnmeani) / (bnstdi + 1e-5) + bnbias  # +1e-5 防止除零

    h = torch.tanh(hpreact)              # Tanh 激活 → (32, 200)
    logits = h @ W2 + b2                 # 输出层 → (32, 27)
    loss = F.cross_entropy(logits, Yb)   # 交叉熵损失

    # ---- 4. 用 EMA 在训练过程中维护全局 mean/std，供推理时使用 ----
    # momentum=0.001: running 值 ≈ 反映最近 ~1000 个 batch 的平均，非常平滑
    # 不参与梯度计算（torch.no_grad），这是簿记操作而非可学习参数
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

    # ---- 5. 反向传播 ----
    for p in parameters:
        p.grad = None                    # 清零梯度（比 zero_() 更高效）
    loss.backward()

    # ---- 6. 参数更新（SGD） ----
    lr = 0.1 if i < 100000 else 0.01     # 学习率衰减：前 100K 步用 0.1，之后用 0.01
    for p in parameters:
        p.data += -lr * p.grad           # 梯度下降：参数 -= lr × 梯度

    # ---- 7. 记录 & 打印 ----
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps}: {loss.item():.4f}')
    lossi.append(loss.item())

## 2. 初始化诊断：为什么初始 loss 很重要？

训练结束后我们会检查三件事来评估初始化质量：

1. **Tanh 饱和度**：有多少神经元的输出 |h| > 0.99？（越少越好）
2. **激活值分布**：h 和 hpreact 的直方图是否合理？
3. **初始 loss**：是否接近理论值 $-\log(1/27) \approx 3.296$？

如果初始 loss 远大于 3.296，说明模型在第一步就"自信地猜错了"——某些类别被赋予了极高概率，而正确答案的概率很低。这通常是因为 W2 的初始化值太大，导致 logits 的绝对值很大，softmax 输出变得极端。

**生活类比：** 一个刚入学的新生被问"你觉得期末考试会考什么？"。合理的回答是"不确定，每个话题都有可能"（均匀分布，loss ≈ 3.3）。不合理的是"我 100% 确定考微积分！"——如果考的不是微积分，这个自信的错误答案会受到很大惩罚（loss 爆炸）。

In [ ]:
# ============================================================
# 可视化 Tanh 饱和情况：白色像素 = |h| > 0.99（神经元"卡死"在 ±1）
# 行 = 样本，列 = 隐藏层神经元。理想情况下应该几乎全黑（很少饱和）
# ============================================================
plt.figure(figsize=(20,10))
plt.imshow(h.abs()>0.99, cmap='gray', interpolation='nearest')

In [ ]:
# Tanh 输出 h 的分布直方图
# 理想：均匀分布在 (-1, 1) 之间；不理想：两端 ±1 处出现尖峰（饱和）
plt.hist(h.view(-1).tolist(), 50);

In [ ]:
# BN 后、Tanh 前的 hpreact 分布直方图
# 经过 BN 后应该接近标准正态分布（均值≈0，标准差≈1）
plt.hist(hpreact.view(-1).tolist(), 50);

In [ ]:
# ============================================================
# 演示：Softmax "过度自信" 问题
# 当 logits 初始值过大时，softmax 会给某一个类极高的概率，
# 对应的 -log(p) 损失就会非常大 → 初始 loss 爆炸
# 解决方案：初始化 W2 时乘以很小的值（如 0.01），让初始 logits ≈ 0
# ============================================================
logits = torch.randn(4) * 0.1            # 模拟 4 类的 logits（很小 → 接近均匀）
probs = torch.softmax(logits, dim=0)     # 转为概率分布
loss = max(-probs.log())                 # 最坏情况下的 NLL
logits, probs, loss

In [ ]:
# 理论最优初始 loss：如果模型完全不确定（27 类均匀分布），loss = -log(1/27)
# ≈ 3.296。如果初始 loss 远大于此，说明初始化有问题（模型"过度自信"地猜错了）
-torch.tensor(1/27).log()

In [ ]:
# 绘制训练过程中的 loss 曲线（每一步记录一个点）
# 应该看到 loss 从 ~3.3 快速下降，最终趋于平稳
plt.plot(lossi)

## 3. BN 推理模式与模型评估

训练时 BN 用的是每个 mini-batch 的统计量（均值、标准差），但推理时我们需要**确定性**的结果——同一个输入每次都应该得到相同的输出。

**两种获取全局统计量的方法：**

1. **训练后校准**（已弃用）：训练结束后，把整个训练集跑一遍前向传播，精确计算全局 mean/std
2. **指数移动平均（EMA）**（推荐）：训练过程中每一步都用 `running = 0.999 × running + 0.001 × batch_stat` 平滑积累

**生活类比：** 这就像计算一个城市的"平均气温"。方法 1 是年底把 365 天的温度全部重新平均一遍——精确但费时。方法 2 是每天更新一次滑动平均——不需要存储历史数据，随时可用，而且在数据量够大时误差极小。PyTorch 的 `nn.BatchNorm1d` 默认使用方法 2。

> **💡 深入理解：** 为什么 momentum 设为 0.001 这么小？
>
> momentum 越小，running 统计量越"平滑"（反映更长时间段的平均），对单个 mini-batch 的波动越不敏感。0.001 意味着大约需要 ~1000 个 batch 才能让 running 值"追上"真实分布。这对于 200K 步的训练来说绰绰有余，但如果训练步数很少，就需要增大 momentum（如 PyTorch 默认的 0.1）。

In [ ]:
# ============================================================
# [已弃用] BN 校准的另一种方法：训练结束后，用整个训练集做一次前向传播
# 来精确计算全局 mean/std。现在改用 EMA（running mean/std）替代。
# 保留此代码供对比参考。
# ============================================================
# # 训练结束后校准 BN：计算整个训练集的全局 mean/std
# # 训练时用的是 mini-batch 的统计量（有噪声），推理时需要稳定的全局统计量
# with torch.no_grad():
#     # 对整个训练集做一次前向传播（不需要梯度）
#     emb=C[Xtr] 
#     embcat=emb.view(emb.shape[0],-1) 
#     hpreact=embcat @ W1 + b1 
#     # 在全部训练样本上计算每个神经元的均值和标准差 → 形状 (1, n_hidden)
#     bnmean=hpreact.mean(0,keepdim=True) 
#     bnstd=hpreact.std(0,keepdim=True)
# # 现在用 bnmean_running 和 bnstd_running 就可以在推理时使用——
# # 不需要单独跑校准步骤，训练过程中 EMA 已经自动维护了全局统计量

In [ ]:
# 查看最后一个 mini-batch 计算的 BN 均值（每个神经元一个值）
# 这是"局部"统计量，每个 batch 都不同
bnmean

In [ ]:
# 查看 EMA 维护的全局 BN 均值（训练过程中平滑积累）
# 对比 bnmean：running 版本更稳定，适合推理时使用
bnmean_running

In [ ]:
# ============================================================
# 评估函数：在训练集/验证集/测试集上计算 loss
# 推理时使用全局 bnmean/bnstd 而非 mini-batch 统计量
# ============================================================
@torch.no_grad()                         # 禁用梯度追踪，节省显存和计算
def spilt_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'dev': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x]                           # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1)  # (N, block_size*n_embd)
    hpreact = embcat @ W1                # (N, n_hidden)，不加 b1

    # 关键：推理时用全局 bnmean/bnstd 而不是当前 batch 的统计量
    # 这保证了同一个输入每次预测结果都一样（确定性推理）
    hpreact = bngain * (hpreact - bnmean) / (bnstd + 1e-5) + bnbias
    h = torch.tanh(hpreact)              # (N, n_hidden)
    logits = h @ W2 + b2                 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(f'{split} loss: {loss.item():.4f}')

spilt_loss('train')
spilt_loss('dev')

In [ ]:
# 检查中间张量的形状，帮助调试维度问题
# embcat: (N, 30), W1: (30, 200), hpreact: (N, 200)
embcat.shape, W1.shape, hpreact.shape

In [ ]:
# ============================================================
# 从训练好的模型中采样生成新名字
# 流程：初始上下文 [...] → 逐字符预测 → 直到生成终止符 '.'
# ============================================================
g = torch.Generator().manual_seed(2147483647+10)

for _ in range(20):                      # 生成 20 个名字
    out = []
    context = [0] * block_size           # 初始上下文全是 '.'
    while True:
        # 前向传播当前上下文
        emb = C[torch.tensor([context])] # (1, block_size, n_embd)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)  # (1, n_hidden)
        logits = h @ W2 + b2            # (1, vocab_size)
        probs = F.softmax(logits, dim=1) # 转为概率分布

        # 按概率分布随机采样下一个字符
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()

        # 滑动上下文窗口 & 记录生成的字符
        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0:                      # 生成了终止符 '.' → 一个名字生成完毕
            break

    print(''.join(itos[i] for i in out)) # 将索引序列解码为字符串并打印

In [ ]:
# ============================================================
# Loss 变化记录：每次改进后的 train/val loss 对比
# 可以清楚看到每个优化技巧的贡献
# ============================================================
#losslog 
# original: 
# train 2.1245  val 2.1682

# fix softmax confidently wrong:       ← W2 *= 0.01，避免初始 loss 爆炸
# train 2.07    val 2.13

# fix tanh layer too saturated at init: ← Kaiming 初始化 W1
# train 2.0356  val 2.1027

# add batchnorm:                       ← 加入 BN 层
# train 2.0786  dev 2.1128

# delete the b1, momentum=0.001        ← 去掉 b1（BN 使其多余）
# train loss: 2.4232  dev loss: 2.4459

## 4. 从浅到深：Kaiming 初始化与模块化网络

前面用一个隐藏层 + 手写 BN 实验了初始化的重要性。现在要进入"正式"模式：

1. **Kaiming 初始化演示**：为什么权重要除以 $\sqrt{fan\_in}$？
2. **用类封装层**：Linear、BatchNorm1d、Tanh —— 和 PyTorch 官方接口一模一样
3. **搭建 5 层深网络**：观察深度带来的新挑战

**直觉理解：** 想象一排人在传话——如果每个人说话的音量都比上一个人大一点，到最后一个人就变成了尖叫（梯度爆炸）；如果每个人的音量都缩小一点，到最后就听不见了（梯度消失）。Kaiming 初始化的目标就是让每个人说话的音量保持一致。

### Kaiming 初始化公式

$$W \sim \mathcal{N}\left(0, \frac{1}{fan\_in}\right)$$

逐项解读：
- $fan\_in$ = 输入维度（一个神经元有多少条输入连接）
- 为什么除以 $fan\_in$？因为矩阵乘法会将 $fan\_in$ 个随机数相加，根据中心极限定理，和的标准差 = 单个值的标准差 × $\sqrt{fan\_in}$
- 如果还用 Tanh 激活，需要额外乘以增益（gain）$\frac{5}{3}$ 来补偿方差压缩

**类比：** Kaiming 初始化就像调音台上的"音量标准化"——不管输入通道有多少条，混音后的总音量始终保持一致。输入通道越多（fan_in 越大），每条通道的音量就要调得越小。

In [ ]:
# ============================================================
# 演示 Kaiming 初始化的原理：为什么要除以 sqrt(fan_in)？
# 如果输入 x ~ N(0,1)，权重 w ~ N(0,1)，那么 y = x @ w 的标准差 = sqrt(fan_in)
# 除以 sqrt(fan_in) 后，输出 y 的标准差重新回到 ≈ 1
# ============================================================
x = torch.randn(1000, 10)               # 1000 个样本，10 维输入（fan_in=10）
w = torch.randn(10, 200) / 10**0.5      # 权重除以 sqrt(10) → Kaiming 初始化
y = x @ w                               # 矩阵乘法 → (1000, 200)
print(x.mean(), x.std())                # 输入：mean≈0, std≈1
print(y.mean(), y.std())                # 输出：mean≈0, std≈1（被 Kaiming 保持住了！）

plt.figure(figsize=(20,5))
plt.subplot(121)                         # 左图：输入 x 的分布
plt.hist(x.view(-1).tolist(), 50, density=True)
plt.subplot(122)                         # 右图：输出 y 的分布（应该也是标准正态）
plt.hist(y.view(-1).tolist(), 50, density=True)

In [ ]:
# ============================================================
# 用类封装的深层网络：5 层隐藏层 + BatchNorm1d
# 架构：Embedding → [Linear → BN → Tanh] × 5 → Linear → Softmax
# 这是"正式"写法，替代前面手写的单层 BN
# ============================================================

class Linear:
    """线性层：y = x @ W + b"""
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / (fan_in**0.5)  # Kaiming 初始化
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight       # 矩阵乘法
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])
    

class BatchNorm1d:
    """批归一化层：训练时用 mini-batch 统计量，推理时用 running 统计量"""
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps                   # 防止除零的小常数
        self.momentum = momentum         # EMA 动量（0.1 → 近 ~10 个 batch 的平均）
        self.training = True             # 训练/推理模式开关

        # 可学习参数（通过反向传播更新）
        self.gamma = torch.ones((1, dim))   # 缩放因子 γ，初始 = 1
        self.beta = torch.zeros((1, dim))   # 偏移量 β，初始 = 0

        # 缓冲区（通过 EMA 更新，不参与梯度计算）
        self.running_mean = torch.zeros((1, dim))
        self.running_var = torch.ones((1, dim))

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)                    # batch 均值
            xvar = x.var(0, keepdim=True, unbiased=False)       # batch 方差（有偏估计）
        else:
            xmean = self.running_mean    # 推理时用全局统计量
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)       # 标准化
        self.out = self.gamma * xhat + self.beta                # 缩放 + 偏移

        # 训练时更新 running 统计量
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]
    

class Tanh:
    """Tanh 激活函数层"""
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []

# ============================================================
# 组装 5 层深层网络
# ============================================================    
n_embd = 10                              # 嵌入维度
n_hidden = 100                           # 每层隐藏神经元数
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)  # 嵌入矩阵 (27, 10)
layers = [
    Linear(block_size*n_embd, n_hidden), Tanh(),  # 第 1 层：30 → 100
    Linear(n_hidden, n_hidden), Tanh(),           # 第 2 层：100 → 100
    Linear(n_hidden, n_hidden), Tanh(),           # 第 3 层：100 → 100
    Linear(n_hidden, n_hidden), Tanh(),           # 第 4 层：100 → 100
    Linear(n_hidden, n_hidden), Tanh(),           # 第 5 层：100 → 100
    Linear(n_hidden, vocab_size),                 # 输出层：100 → 27（无激活）
]

with torch.no_grad():
    # 输出层权重 × 0.1 → 初始 logits 接近 0 → 均匀分布 → 避免初始 loss 爆炸
    layers[-1].weight *= 0.1
    # 其他 Linear 层权重 × 5/3 → 补偿 Tanh 的方差压缩
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5/3          # gain = 5/3 ≈ 1/0.6，刚好抵消 tanh 的压缩

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters), 'parameters')
for p in parameters:
    p.requires_grad = True

### 为什么增益设为 `5/3`？—— Tanh 的方差压缩效应

`layer.weight *= 0.5` 或 `layer.weight *= 5/3` 这行代码的目的是控制每层激活值的方差。

**Tanh 会"压缩"方差：** tanh 的输出总是比输入小一点——靠近 0 的值几乎不变（斜率 ≈ 1），但离 0 越远的值被压得越多（曲线变平）。当输入服从标准正态分布时，经过 tanh 后方差大约降到原来的 **3/5**（≈ 0.6）。

**逐层累积效应：** 如果不补偿这个压缩，每过一层方差都乘以 0.6，5 层之后方差只剩 0.6⁵ ≈ 0.08，激活值全部坍缩到 0 附近。如果增益设得太小（如 0.5），情况更严重——方差每层乘以 0.25，5 层后几乎归零。

**5/3 是 3/5 的倒数**，刚好补偿 tanh 的压缩，让每层输出方差保持稳定：

| 增益值 | 每层方差变化 | 5层后效果 |
|--------|------------|----------|
| 0.5 | ×0.25 | 激活坍缩到 0 |
| 1.0 | ×0.6 | 缓慢衰减 |
| **5/3** | **≈×1.0** | **保持稳定 ✓** |
| 2.0 | ×2.4 | 激活饱和到 ±1 |

> 这个值也是 PyTorch 官方推荐的：`torch.nn.init.calculate_gain('tanh')` 返回的就是 `5/3`。

In [ ]:
# ============================================================
# 训练深层网络（5 层 MLP，无 BN）
# 注意：这里没有使用 BatchNorm1d，是为了对比观察初始化的效果
# ============================================================
max_steps = 200000
batch_size = 32
lossi = []
ud = []                                  # 记录 update-to-data 比率（诊断用）

for i in range(max_steps):
    # ---- 构造 mini-batch ----
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # ---- 前向传播：逐层传递 ----
    emb = C[Xb]                          # (32, 3, 10)
    x = emb.view(emb.shape[0], -1)      # (32, 30)
    for layer in layers:
        x = layer(x)                     # 依次通过每一层
    loss = F.cross_entropy(x, Yb)        # x 此时是 logits (32, 27)

    # ---- 反向传播 ----
    for layer in layers:
        layer.out.retain_grad()          # 保留中间激活的梯度，用于后续可视化
    for p in parameters:
        p.grad = None
    loss.backward()

    # ---- SGD 更新 ----
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # ---- 记录统计量 ----
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps}: {loss.item():.4f}')
    lossi.append(loss.item())

    # 记录每个参数的 update/data 比率 → 诊断学习率是否合适
    with torch.no_grad():
        ud.append([lr*p.grad.std()/p.data.std().log10().item() for p in parameters])

    if i > 1000:
        break                            # 只跑 1000 步用于快速诊断

## 5. 训练诊断四件套

训练一个深层网络，光看 loss 曲线是远远不够的。Karpathy 在视频中反复强调：**你需要看网络内部在发生什么**。下面的四张诊断图是他推荐的"健康检查"工具包：

| 诊断图 | 看什么 | 健康信号 | 危险信号 |
|--------|--------|---------|---------|
| 激活值分布 | 每层 Tanh 输出的直方图 | 各层形状、宽度一致 | 逐层收窄（坍缩）或挤到 ±1（饱和） |
| 激活梯度分布 | 每层 ∂loss/∂h 的直方图 | 各层量级相近 | 逐层缩小（消失）或放大（爆炸） |
| 权重梯度分布 | 每个 W 矩阵梯度的直方图 | 量级在 1e-3 附近 | 某层梯度为 0 或极大 |
| update/data 比率 | `lr × grad.std() / data.std()` | 稳定在 ~1e-3（log₁₀ ≈ -3） | 远大于 -3 → 学习率太大；远小于 → 太小 |

**类比：** 这四张图就像医院的体检报告——loss 曲线只是"体温"，告诉你"有没有发烧"，但无法定位是哪个器官出了问题。激活值分布是"血压"，梯度分布是"血常规"，update/data 比率是"代谢指标"。只有四张图一起看，才能判断网络的训练状态是否健康。

> **💡 深入理解：** `retain_grad()` 的作用
>
> PyTorch 默认只保留叶子节点（`requires_grad=True` 的参数）的梯度，中间计算结果的梯度会被丢弃以节省显存。调用 `layer.out.retain_grad()` 告诉 PyTorch 保留这些中间激活的梯度，这样我们才能可视化每一层的梯度流动情况。生产代码中不需要这样做——这纯粹是诊断工具。

In [ ]:
# ============================================================
# 诊断图 1：各层 Tanh 激活值的分布
# 观察要点：
#   - 分布是否逐层收窄（方差坍缩） → 说明增益不足
#   - 分布是否挤到 ±1（饱和） → 说明增益过大
#   - 理想：各层分布形状和宽度大致相同
# ============================================================
plt.figure(figsize=(20,6))
legend = []

for i, layer in enumerate(layers[:-1]): # 遍历除输出层外的所有层
    if isinstance(layer, Tanh):
        t = layer.out                    # 该层 Tanh 的输出
        print('layer %d (%10s): mean %+.2f, std %.2f, saturated %.2f%%' % (
            i, layer.__class__.__name__, t.mean(), t.std(), (t.abs()>0.97).float().mean()*100
        ))
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legend.append(f'layer{i} {layer.__class__.__name__}')
plt.legend(legend)
plt.title('activation distributions')

In [ ]:
# ============================================================
# 诊断图 2：各层 Tanh 激活值的**梯度**分布
# 观察要点：
#   - 梯度是否逐层变小（梯度消失） → 深层学不到东西
#   - 梯度是否逐层变大（梯度爆炸） → 训练不稳定
#   - 理想：各层梯度的量级大致相同
# ============================================================
plt.figure(figsize=(20,6))
legend = []
for i, layer in enumerate(layers[:-1]):
    if isinstance(layer, Tanh):
        t = layer.out.grad               # 该层 Tanh 输出的梯度 ∂loss/∂h
        print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))   
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legend.append(f'layer {i} {layer.__class__.__name__}')
plt.legend(legend)
plt.title('gradient distributions')

In [ ]:
# ============================================================
# 诊断图 3：各权重矩阵的**梯度**分布
# 只看 2D 权重（跳过 1D 偏置），因为权重梯度更能反映学习动态
# grad:data ratio 越接近 1e-3 越好（Karpathy 经验法则）
# ============================================================
plt.figure(figsize=(20,6))
legend = []
for i, p in enumerate(parameters):
    t = p.grad
    if p.ndim == 2:                      # 只看权重矩阵，不看偏置
        print('weight %10s | mean %+f | std %e | grad:data ratio: %e' % (
            tuple(p.shape), t.mean(), t.std(), t.abs().mean()/loss.item()))
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legend.append(f'param {i} {tuple(p.shape)}')

plt.legend(legend)
plt.title('weight gradient distributions')

In [ ]:
# ============================================================
# 诊断图 4：update-to-data 比率随训练步数的变化
# 公式：ratio = lr × grad.std() / data.std()
# 经验法则：ratio ≈ 1e-3（图中 log10 = -3 的水平线）
#   - 远大于 -3 → 学习率过大，参数剧烈震荡
#   - 远小于 -3 → 学习率过小，参数几乎不动
# ============================================================
plt.figure(figsize=(20, 4))
legends = [] 
for i, p in enumerate(parameters):
    if p.ndim == 2:                      # 只看权重矩阵
        plt.plot([ud[j][i] for j in range(len(ud))])
        legends.append('param%d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k')   # 画 -3 参考线
plt.legend(legends)